In [2]:
import datacube, pandas as pd, re
dc = datacube.Datacube()

prods = dc.list_products()
names = pd.Index(prods.index.astype(str))

KW = [
    "salinity", "salt", "ec", "conductivity",
    "turbidity", "tss", "tsm", "spm",
    "chlorophyll", "chl", "cdom", "doc",
    "water_quality", "aquawatch", "acolite",
    "sentinel-2", "s2", "landsat"
]

pat = re.compile("|".join([re.escape(k) for k in KW]), re.IGNORECASE)
hits = [n for n in names if pat.search(n) or pat.search(str(prods.loc[n].to_dict()))]

print("Candidate products:")
for h in sorted(hits):
    print(" -", h, "|", prods.loc[h].get("description", ""))


Candidate products:
 - chl | Chlorophyll a concentration retrieved from Landsat-8 OLI
 - dl_rs_mda | Coastal Water Quality Maps derived from NASA MODIS-Aqua L2 Ocean Color (nasa_aqua_l2_oc) imagery using a deep learning model trained on an augmented in-situ based spectral library (DL-RS_V1)
 - dl_wq_ls8 | Dense Deep Learning Model-derived Total Suspended Sediments (TSS), Dissolved Organic Carbon (DOC) retrieved from Landsat-8
 - doc | Dissolved organic carbon retrieved from Landsat-8 OLI
 - enmap_hsi_l2a | EnMAP HSI - Level 2A Hyperspectral Images
 - ga_ls_mangrove_cover_cyear_3 | Geoscience Australia Landsat Mangrove Cover Calendar Year Collection 3
 - jpss2_viirs_l2_iop_nrt | NASA NOAA-21 L2 NRT Inherent Optical Properties, regridded to WGS84 0.0067 x 0.0067 deg
 - jpss2_viirs_l2_oc_nrt | NASA NOAA-21 L2 NRT Ocean Color, regridded to WGS84 0.0067 x 0.0067 deg
 - landsat5_c2l2_sr | Landsat 5 Collection 2 Level-2 Surface Reflectance Product. 30m UTM based projection.
 - landsat5_c2l2_s

In [3]:
import re
import datacube

dc = datacube.Datacube()

def list_landsat_st_products(dc):
    prods = dc.list_products()
    # Columnas típicas: name, description, ... (depende del index)
    names = list(prods.index)

    cand = []
    for p in names:
        p_low = p.lower()
        if ("landsat" in p_low) and ("c2" in p_low) and ("l2" in p_low) and ("st" in p_low):
            cand.append(p)

    # Fallback: si el naming no incluye 'st', revisa measurements
    # (ojo: list_measurements puede ser pesado si hay muchos productos)
    if not cand:
        meas = dc.list_measurements()
        for p in names:
            try:
                m = meas.loc[p]
                mnames = set(m.index.astype(str).str.lower())
                if "lwir11" in mnames and "qa_pixel" in mnames:
                    cand.append(p)
            except Exception:
                pass

    return sorted(set(cand))

st_products = list_landsat_st_products(dc)
print("Detected Landsat ST products:", st_products)


Detected Landsat ST products: ['landsat5_c2l2_st', 'landsat7_c2l2_st', 'landsat8_c2l2_st', 'landsat9_c2l2_st']


In [2]:
## MY CREDENTIALS ##

import os
print("AWS_ACCESS_KEY_ID present?:", "AWS_ACCESS_KEY_ID" in os.environ)
print("AWS_PROFILE present?:", "AWS_PROFILE" in os.environ)
print("AWS_SHARED_CREDENTIALS_FILE:", os.environ.get("AWS_SHARED_CREDENTIALS_FILE"))

import os

# Requester Pays (necesario para ese bucket)
os.environ["AWS_REQUEST_PAYER"] = "requester"

# NO uses "no sign" aquí; quítalo si estaba
os.environ.pop("AWS_NO_SIGN_REQUEST", None)

# A veces ayuda fijar región
os.environ.setdefault("AWS_DEFAULT_REGION", "us-west-2")
os.environ.setdefault("AWS_REGION", "us-west-2")


AWS_ACCESS_KEY_ID present?: True
AWS_PROFILE present?: False
AWS_SHARED_CREDENTIALS_FILE: None


'us-west-2'

In [27]:
import os

def is_set(k): 
    v = os.environ.get(k)
    return (v is not None) and (len(v.strip()) > 0)

print("AWS_ACCESS_KEY_ID:", is_set("AWS_ACCESS_KEY_ID"))
print("AWS_SECRET_ACCESS_KEY:", is_set("AWS_SECRET_ACCESS_KEY"))
print("AWS_SESSION_TOKEN:", is_set("AWS_SESSION_TOKEN"))  # importante si son credenciales temporales

print("AWS_DEFAULT_REGION:", os.environ.get("AWS_DEFAULT_REGION"))
print("AWS_REGION:", os.environ.get("AWS_REGION"))

print("AWS_REQUEST_PAYER:", os.environ.get("AWS_REQUEST_PAYER"))
print("AWS_NO_SIGN_REQUEST:", os.environ.get("AWS_NO_SIGN_REQUEST"))


AWS_ACCESS_KEY_ID: True
AWS_SECRET_ACCESS_KEY: True
AWS_SESSION_TOKEN: True
AWS_DEFAULT_REGION: us-west-2
AWS_REGION: us-west-2
AWS_REQUEST_PAYER: requester
AWS_NO_SIGN_REQUEST: None


In [48]:
import os

# --- AWS (Requester Pays) ---
os.environ["AWS_REQUEST_PAYER"] = "requester"
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
os.environ["AWS_REGION"] = "us-west-2"
os.environ.pop("AWS_NO_SIGN_REQUEST", None)

# --- GDAL / VSI S3 stability ---
# Evita que GDAL intente listar directorios/prefijos (ListBucket) y falle con 403
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "YES"

# Reduce “probing” extraño; ayuda a que vaya directo por objetos .TIF
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.TIF,.xml,.XML"
os.environ["VSI_CACHE"] = "TRUE"
os.environ["VSI_CACHE_SIZE"] = str(128 * 1024 * 1024)  # 128MB


In [49]:
## 0 ##
# Config + utilities (run once)
# - Paths/constants
# - AOI bbox (WGS84) for ODC searches
# - AOI geometry for true polygon clip in OUTPUT_CRS
# - Landsat ST product discovery
# - Metadata extraction (datetime, wrs path/row)
# - QA masking + ST→Celsius
# - Measurement resolver by aliases (no hardcode)


from pathlib import Path
import os
import re
import uuid
import geopandas as gpd
import pandas as pd
import datacube
import xarray as xr

# ----------------------------
# Paths
# ----------------------------
INPUTS_DIR  = Path("inputs")
OUTPUTS_DIR = Path("outputs")

AOI_FILE    = INPUTS_DIR / "delta_legal_4326.geojson"
TARGETS_CSV = INPUTS_DIR / "target_dates.csv"

# outputs for pair-selection stage
OUT_COVERAGE_PAIRS_CSV = OUTPUTS_DIR / "satellite_coverage_report_pairs.csv"
OUT_EFFECTIVE_TILES_CSV = INPUTS_DIR / "target_dates_effective_tiles.csv"

# ----------------------------
# Spatial + selection settings
# ----------------------------
OUTPUT_CRS = "EPSG:32610"
RESOLUTION = (-30, 30)

# Must-have WRS for full Delta legal coverage
WRS_PATH = 44
WRS_ROWS = {33, 34}

# Search window around each target date
SEARCH_WINDOW_DAYS = 16

# You said: do NOT use cloud cover filter yet
CLOUD_COVER_MAX = 18  # keep None until you want it

# Masking behavior
USE_CLEAR  = False
WATER_ONLY = True
USE_RADSAT_MASK = False  # optional (if qa_radsat exists)

# Optional tiebreaker preference (only used if offsets are identical)
# You can reorder or set to [] if you truly want no preference
SENSOR_PREF = ["landsat9", "landsat8", "landsat7", "landsat5", "landsat4"]

# ----------------------------
# AOI helpers
# ----------------------------
_AOI_CACHE_4326 = None

def load_aoi_4326(aoi_path: Path = AOI_FILE) -> gpd.GeoDataFrame:
    global _AOI_CACHE_4326
    if _AOI_CACHE_4326 is not None:
        return _AOI_CACHE_4326

    if not aoi_path.exists():
        raise FileNotFoundError(f"AOI file not found: {aoi_path.resolve()}")

    aoi = gpd.read_file(aoi_path)
    if aoi.crs is None:
        raise ValueError("AOI has no CRS defined (aoi.crs is None).")

    # Ensure EPSG:4326
    if aoi.crs.to_string() != "EPSG:4326":
        aoi = aoi.to_crs("EPSG:4326")

    # Basic validity check
    if (~aoi.is_valid).any():
        # Fix typical issues
        aoi["geometry"] = aoi.geometry.buffer(0)
        aoi = aoi[aoi.geometry.notnull()].copy()

    # Ensure single feature (you already have 1, but keep robust)
    if len(aoi) > 1:
        aoi = aoi.dissolve().reset_index(drop=True)

    _AOI_CACHE_4326 = aoi
    return aoi

def get_bbox_wgs84(aoi_path: Path = AOI_FILE) -> dict:
    """bbox in WGS84 for datacube.find_datasets/load: {'x':(minx,maxx), 'y':(miny,maxy)}"""
    aoi = load_aoi_4326(aoi_path)
    minx, miny, maxx, maxy = aoi.total_bounds
    bbox = {"x": (minx, maxx), "y": (miny, maxy)}
    print("AOI bbox (EPSG:4326):", bbox)
    return bbox

def get_aoi_in_crs(dst_crs: str = OUTPUT_CRS, aoi_path: Path = AOI_FILE) -> gpd.GeoDataFrame:
    """AOI geometry reprojected to dst_crs (used for true polygon clip)."""
    aoi = load_aoi_4326(aoi_path)
    return aoi.to_crs(dst_crs)

# ----------------------------
# Time helpers
# ----------------------------
def _date_range(center_date, days):
    c = pd.Timestamp(center_date).normalize()
    return (c - pd.Timedelta(days=days), c + pd.Timedelta(days=days + 1))  # end exclusive

def _as_utc_naive(ts):
    ts = pd.Timestamp(ts)
    if ts.tz is not None:
        return ts.tz_convert("UTC").tz_localize(None)
    return ts

def _extract_scene_datetime(ds):
    md = getattr(ds, "metadata_doc", None) or {}
    # STAC-style: properties.datetime
    if isinstance(md, dict):
        dt = (md.get("properties", {}) or {}).get("datetime", None)
        if dt:
            try:
                return _as_utc_naive(pd.to_datetime(dt))
            except Exception:
                pass
    # fallbacks
    ct = getattr(ds, "center_time", None)
    if ct is not None:
        try:
            return _as_utc_naive(pd.to_datetime(ct))
        except Exception:
            pass
    try:
        return _as_utc_naive(pd.to_datetime(ds.metadata.time))
    except Exception:
        return None

def _extract_cloud_cover(ds):
    md = getattr(ds, "metadata_doc", None) or {}
    if isinstance(md, dict):
        cc = (md.get("properties", {}) or {}).get("eo:cloud_cover", None)
        if cc is None:
            return None
        try:
            return float(cc)
        except Exception:
            return None
    return None

# ----------------------------
# WRS path/row extraction (robust)
# ----------------------------
_SCENEID_PR = re.compile(r".*_(\d{3})(\d{3})_.*")  # ..._PPP RRR_...

def _extract_wrs_path_row(ds):
    """
    Tries to get WRS path/row from STAC properties.
    Fallback: parse landsat:scene_id like LC09_L2SP_044033_YYYYMMDD_...
    Returns (path:int|None, row:int|None).
    """
    md = getattr(ds, "metadata_doc", None) or {}
    props = {}
    if isinstance(md, dict):
        props = (md.get("properties", {}) or {})

    # Common STAC keys
    for kpath, krow in [
        ("landsat:wrs_path", "landsat:wrs_row"),
        ("wrs_path", "wrs_row"),
        ("landsat:wrs_path", "wrs_row"),
        ("wrs_path", "landsat:wrs_row"),
    ]:
        p = props.get(kpath, None)
        r = props.get(krow, None)
        try:
            p = int(p) if p is not None else None
            r = int(r) if r is not None else None
        except Exception:
            p, r = None, None
        if p is not None and r is not None:
            return p, r

    # Fallback: parse scene_id
    scene_id = props.get("landsat:scene_id", None) or props.get("scene_id", None)
    if scene_id:
        m = _SCENEID_PR.match(str(scene_id))
        if m:
            try:
                return int(m.group(1)), int(m.group(2))
            except Exception:
                pass

    return None, None

# ----------------------------
# Landsat ST product discovery
# ----------------------------
def _product_priority(product_name: str) -> int:
    """
    Used only as a tiebreaker AFTER we find a valid 2-tile pair.
    Lower number = preferred.
    """
    p = product_name.lower()
    for i, key in enumerate(SENSOR_PREF, start=1):
        if key in p:
            return i
    return 999

def list_landsat_st_products(dc: datacube.Datacube):
    prods = dc.list_products()
    names = list(prods.index.astype(str))

    # Prefer naming-based detection first
    cand = []
    for name in names:
        n = name.lower()
        if ("landsat" in n) and ("c2" in n) and ("l2" in n) and ("st" in n):
            cand.append(name)

    # Fallback: detect by measurements if naming doesn't match
    if not cand:
        meas = dc.list_measurements()
        for name in names:
            try:
                mnames = set(meas.loc[name].index.astype(str).str.lower())
                if "qa_pixel" in mnames and ("lwir11" in mnames or "st" in mnames):
                    cand.append(name)
            except Exception:
                pass

    return sorted(set(cand), key=lambda x: (_product_priority(x), x))

# ----------------------------
# QA mask + ST conversion
# ----------------------------
BIT_DILATED_CLOUD = 1
BIT_CIRRUS        = 2
BIT_CLOUD         = 3
BIT_CLOUD_SHADOW  = 4
BIT_SNOW          = 5
BIT_CLEAR         = 6
BIT_WATER         = 7

def _bit_is_set(x, bit):
    return (x.astype("uint16") & (1 << bit)) > 0

def build_good_mask(qa_pixel: xr.DataArray) -> xr.DataArray:
    bad = (
        _bit_is_set(qa_pixel, BIT_DILATED_CLOUD) |
        _bit_is_set(qa_pixel, BIT_CIRRUS) |
        _bit_is_set(qa_pixel, BIT_CLOUD) |
        _bit_is_set(qa_pixel, BIT_CLOUD_SHADOW) |
        _bit_is_set(qa_pixel, BIT_SNOW)
    )
    good = ~bad
    if USE_CLEAR:
        good = good & _bit_is_set(qa_pixel, BIT_CLEAR)
    if WATER_ONLY:
        good = good & _bit_is_set(qa_pixel, BIT_WATER)
    return good

def st_to_celsius(st_da: xr.DataArray) -> xr.DataArray:
    return st_da - 273.15  # Kelvin -> Celsius

# ----------------------------
# Measurement resolver (aliases-driven)
# ----------------------------
def _aliases_lower(meas_def: dict) -> set:
    als = meas_def.get("aliases", []) or []
    return {str(a).lower() for a in als}

def resolve_measurements_from_definition(dc: datacube.Datacube, product: str):
    """
    Resolve measurement names using product.definition['measurements'] and aliases.
    Returns: (st_name, qa_pixel_name, qa_radsat_name_or_None)
    """
    prod = dc.index.products.get_by_name(product)
    if prod is None:
        raise ValueError(f"Product not found in ODC index: {product}")

    meas_defs = (prod.definition or {}).get("measurements", []) or []
    if not meas_defs:
        raise ValueError(f"No measurements found in product definition for: {product}")

    # 1) Find ST
    st_candidates = []
    for m in meas_defs:
        name = str(m.get("name", ""))
        if not name:
            continue
        als = _aliases_lower(m)

        is_st = (
            ("st" in als) or
            ("surface_temperature" in als) or
            any(a.startswith("st_b") for a in als)
        )
        if is_st:
            rank = 0
            if "st" in als: rank += 100
            if "surface_temperature" in als: rank += 90
            if any(a.startswith("st_b") for a in als): rank += 80
            if "lwir" in name.lower(): rank += 10
            st_candidates.append((rank, name, als))

    if not st_candidates:
        # fallback by units (Kelvin)
        for m in meas_defs:
            name = str(m.get("name", ""))
            units = str(m.get("units", "")).lower()
            if "kelvin" in units and "qa" not in name.lower():
                st_candidates.append((1, name, _aliases_lower(m)))

    if not st_candidates:
        raise KeyError(f"Could not resolve ST measurement for product={product}")

    st_candidates.sort(reverse=True)
    st_name = st_candidates[0][1]

    # 2) qa_pixel
    qa_pixel_name = None
    for m in meas_defs:
        name = str(m.get("name", ""))
        if not name:
            continue
        als = _aliases_lower(m)
        if (name.lower() == "qa_pixel") or ("qa_pixel" in als) or ("pixel_quality" in als) or ("pq" in als):
            qa_pixel_name = name
            break
    if qa_pixel_name is None:
        raise KeyError(f"Could not resolve qa_pixel measurement for product={product}")

    # 3) qa_radsat (optional)
    qa_radsat_name = None
    for m in meas_defs:
        name = str(m.get("name", ""))
        if not name:
            continue
        als = _aliases_lower(m)
        if (name.lower() == "qa_radsat") or ("qa_radsat" in als) or ("radiometric_saturation" in als) or ("radsat" in als):
            qa_radsat_name = name
            break

    return st_name, qa_pixel_name, qa_radsat_name


In [50]:
## 1 ##

# Defines a helper function (find_outflow_col) that automatically detects
# the correct Net Delta Outflow column name in any dataset, even if the column
# name changes (e.g., “NDOI”, “Outflow”, “Net Delta Outflow Index”, “QOUT”,
# “OUT/OUT1”, etc.).
# It standardizes column names, checks for exact or similar matches,
# and returns the correct original column name.

import re
import difflib

def find_outflow_col(df):
    """Returns the ORIGINAL name of the column that contains Net Delta Outflow."""
    
    def _norm(s: str) -> str:
        s = str(s).strip().upper()
        s = re.sub(r"\s+", "_", s)
        s = re.sub(r"[^A-Z0-9_]", "", s)  # removes characters like ()-/.
        return s

    # Accepted aliases (normalized)
    ALIASES = {
        "NDOI", "NDOI_CFS",
        "QOUT",
        "OUT1", "OUT", "OUT2",
        "OUTFLOW", "OUTFLOW_CFS",
        "NET_DELTA_OUTFLOW", "NET_DELTA_OUTFLOW_INDEX",
        "NETDELTAOUTFLOW", "NETDELTAOUTFLOWINDEX"
    }

    # Mapping: original column name -> normalized name
    norm_map = {c: _norm(c) for c in df.columns}

    # 1) Exact match
    exact = [orig for orig, n in norm_map.items() if n in ALIASES]
    if exact:
        return exact[0]

    # 2) Tolerant regex match
    pat = re.compile(
        r"^(NDOI(_CFS)?|QOUT|OUT1|OUT|OUTFLOW(_CFS)?|NET_?DELTA_?OUTFLOW(_INDEX)?)$"
    )
    for orig, n in norm_map.items():
        if pat.match(n):
            return orig

    # 3) Fuzzy matching
    choices = list(ALIASES)
    for orig, n in norm_map.items():
        if difflib.get_close_matches(n, choices, n=1, cutoff=0.8):
            return orig

    raise KeyError(
        f"Outflow column not found. Available columns: {list(df.columns)}"
    )


In [51]:
## 2 ##
# AUTO Dayflow downloader (1929 -> most recent available)
# Output kept compatible with the rest of your pipeline:
#   outputs/dayflow_1929_2024.csv  (Date, NDOI)
# Also writes:
#   outputs/dayflow_1929_present.csv (same content; nicer name)

import io, os, sys, re, requests
import pandas as pd

# --- CKAN endpoints (CNRA Open Data) ---
API_PACKAGE_SHOW = "https://data.cnra.ca.gov/api/3/action/package_show"
API_DS           = "https://data.cnra.ca.gov/api/3/action/datastore_search"
API_RSRC_SHOW    = "https://data.cnra.ca.gov/api/3/action/resource_show"
UA_HDR           = {"User-Agent": "Mozilla/5.0"}

# --- Dataset id/slug in CNRA CKAN ---
PACKAGE_ID = "dayflow"

# --- Output paths (KEEP the old name for compatibility) ---
os.makedirs("outputs", exist_ok=True)
OUT_CSV = os.path.join("outputs", "dayflow_1929_present.csv")


# ---------- helper: builds or detects Date ----------
def build_date_series(df: pd.DataFrame) -> pd.Series:
    cols = {str(c).strip().lower(): c for c in df.columns}

    # common: Date
    if "date" in cols:
        s = pd.to_datetime(df[cols["date"]], errors="coerce")
        if s.notna().any():
            return s

    # fallback: Year / Month / Day
    has = {k: v for k, v in cols.items() if k in ("year", "month", "day")}
    if {"year", "month", "day"}.issubset(has):
        return pd.to_datetime(
            dict(
                year=df[has["year"]],
                month=df[has["month"]],
                day=df[has["day"]],
            ),
            errors="coerce",
        )

    raise KeyError(
        f"Could not find a 'Date' column or (Year, Month, Day). Headers: {list(df.columns)}"
    )

# ---------- discover Dayflow Results resources automatically ----------
print("⇢ Discovering Dayflow resources from CNRA CKAN…")

pkg = requests.get(API_PACKAGE_SHOW, params={"id": PACKAGE_ID}, headers=UA_HDR, timeout=60).json()
if not pkg.get("success"):
    raise RuntimeError(pkg.get("error") or "package_show failed")

resources = pkg["result"].get("resources", [])
if not resources:
    raise RuntimeError("No resources found in CKAN package 'dayflow'")

def _is_results_resource(r: dict) -> bool:
    name = str(r.get("name") or r.get("title") or "").lower()
    fmt  = str(r.get("format") or "").lower()
    # keep only "Dayflow Results ..." (exclude monthly totals / comments / docs)
    if "dayflow results" not in name:
        return False
    if any(bad in name for bad in ["monthly", "totals", "comments", "documentation", "doc", "pdf"]):
        return False
    # accept csv/xlsx/excel or empty (some CKAN entries omit format)
    return (fmt in ["csv", "xlsx", "excel"] or fmt == "")

results_res = [r for r in resources if _is_results_resource(r)]
if not results_res:
    # fallback: be a bit more tolerant
    results_res = [r for r in resources if "results" in str(r.get("name","")).lower()]

print(f"✓ Found {len(results_res)} 'Dayflow Results' resources")

# sort by inferred year if present, else by name
def _infer_year(r: dict):
    txt = f"{r.get('name','')} {r.get('title','')}"
    m = re.search(r"(19\d{2}|20\d{2})", txt)
    return int(m.group(1)) if m else -1

results_res = sorted(results_res, key=lambda r: (_infer_year(r), str(r.get("name",""))))

# ---------- download + standardize each resource ----------
frames = []

for r in results_res:
    rid  = r.get("id")
    name = r.get("name") or r.get("title") or rid
    fmt  = str(r.get("format") or "").upper()

    print(f"\n⇢ Processing: {name}  [format={fmt if fmt else 'UNKNOWN'}]")

    df_raw = None

    # A) Try DataStore (works for many CSV resources)
    try:
        js = requests.get(
            API_DS,
            params={"resource_id": rid, "limit": 50000},
            headers=UA_HDR,
            timeout=60,
        ).json()
        if js.get("success") and "records" in js.get("result", {}):
            df_raw = pd.DataFrame(js["result"]["records"])
            if df_raw.empty:
                df_raw = None
    except Exception:
        df_raw = None

    # B) If not in DataStore, fetch via resource_show + direct download
    if df_raw is None:
        meta = requests.get(API_RSRC_SHOW, params={"id": rid}, headers=UA_HDR, timeout=60).json()
        if not meta.get("success"):
            print(f"   ✖ resource_show failed for {name}")
            continue

        url = meta["result"].get("url")
        if not url:
            print(f"   ✖ No URL for {name}")
            continue

        raw = requests.get(url, headers=UA_HDR, timeout=180).content

        # try Excel then CSV
        try:
            df_raw = pd.read_excel(io.BytesIO(raw), engine="openpyxl")
        except Exception:
            try:
                df_raw = pd.read_csv(io.BytesIO(raw))
            except Exception as e:
                print(f"   ✖ Could not read as Excel/CSV: {e}")
                continue

    # --- detect Date + Outflow, reuse chunk 1's find_outflow_col(df_raw) ---
    try:
        date_series = build_date_series(df_raw)
        outcol = find_outflow_col(df_raw)  # <-- Chunk 1 function
    except Exception as e:
        print(f"   ⚠ Unexpected headers; skipping. → {e}")
        print("      Headers:", list(df_raw.columns))
        continue

    df = (
        pd.DataFrame(
            {
                "Date": pd.to_datetime(date_series, errors="coerce"),
                "NDOI": pd.to_numeric(df_raw[outcol], errors="coerce"),
            }
        )
        .dropna(subset=["Date"])
        .sort_values("Date")
    )

    frames.append(df)
    print(f"   ✓ Rows kept: {len(df):,} (max Date: {df['Date'].max().date()})")

if not frames:
    sys.exit("❌ No Dayflow Results resources could be processed.")

# ---------- concatenate + export ----------
all_df = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates("Date")
    .sort_values("Date")
)

all_df.to_csv(OUT_CSV, index=False)

print(f"\n✔ CSV generated: {OUT_CSV} — rows: {len(all_df):,} — last date: {all_df['Date'].max()}")


⇢ Discovering Dayflow resources from CNRA CKAN…
✓ Found 13 'Dayflow Results' resources

⇢ Processing: Dayflow Results 1929 - 1939  [format=CSV]
   ✓ Rows kept: 3,744 (max Date: 1939-12-31)

⇢ Processing: Dayflow Results 1940 - 1949  [format=CSV]
   ✓ Rows kept: 3,653 (max Date: 1949-12-31)

⇢ Processing: Dayflow Results 1950 - 1955  [format=CSV]
   ✓ Rows kept: 2,099 (max Date: 1955-09-30)

⇢ Processing: Dayflow Results 1956 - 1969  [format=CSV]
   ✓ Rows kept: 5,114 (max Date: 1969-09-30)

⇢ Processing: Dayflow Results 1970 - 1983  [format=CSV]
   ✓ Rows kept: 5,113 (max Date: 1983-09-30)

⇢ Processing: Dayflow Results 1984 - 1996  [format=CSV]
   ✓ Rows kept: 4,749 (max Date: 1996-09-30)

⇢ Processing: Dayflow Results 1997 - 2023  [format=CSV]
   ✓ Rows kept: 9,861 (max Date: 2023-09-30)

⇢ Processing: Dayflow Results 2019  [format=CSV]


/tmp/ipykernel_76/1181224511.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  s = pd.to_datetime(df[cols["date"]], errors="coerce")


   ✓ Rows kept: 365 (max Date: 2019-09-30)

⇢ Processing: Dayflow Results 2020  [format=CSV]
   ✓ Rows kept: 366 (max Date: 2020-09-30)

⇢ Processing: Dayflow Results 2021  [format=CSV]
   ✓ Rows kept: 365 (max Date: 2021-09-30)

⇢ Processing: Dayflow Results 2022  [format=CSV]
   ✓ Rows kept: 365 (max Date: 2022-09-30)

⇢ Processing: Dayflow Results 2023  [format=CSV]
   ✓ Rows kept: 365 (max Date: 2023-09-30)

⇢ Processing: Dayflow Results 2024  [format=XLSX]
   ✓ Rows kept: 366 (max Date: 2024-09-30)

✔ CSV generated: outputs/dayflow_1929_present.csv — rows: 34,699 — last date: 2024-09-30 00:00:00


In [52]:
## 3 ##

# Downloads the official Water Year Type records (1906–present)
# for the Sacramento and San Joaquin Valleys from California’s Data Portal.
# Reshapes the data into wide format (WY, Sac_Type, SJV_Type)
# and saves it as outputs/water_year_type.csv


import requests, pandas as pd, os

RID = "105614f4-c71d-4191-b1f9-ea510afd8b62"
API = "https://data.ca.gov/api/3/action/datastore_search"

def get_all_records(rid):
    records, start, rows = [], 0, 50000
    while True:
        js = requests.get(
            API,
            params={"resource_id": rid, "limit": rows, "offset": start},
            timeout=60
        ).json()
        if not js.get("success"):
            raise RuntimeError(js.get("error"))
        recs = js["result"]["records"]
        records.extend(recs)
        if len(recs) < rows:
            break
        start += rows
    return pd.DataFrame(records)

# 1) download long table
long = get_all_records(RID)

# normalize headers
long.columns = [c.strip() for c in long.columns]

# 2) pivot → wide format
wide = (long.pivot(index="WY", columns="Area", values="WYT")
            .reset_index()
            .rename(columns={
                "Sacramento Valley":  "Sac_Type",
                "San Joaquin Valley": "SJV_Type"}))

wide["WY"] = wide["WY"].astype(int)  # 2000.0 → 2000

# 3) save
os.makedirs("outputs", exist_ok=True)
wide.to_csv("outputs/water_year_type.csv", index=False)
print("✓ outputs/water_year_type.csv — rows:", len(wide))


✓ outputs/water_year_type.csv — rows: 124


In [53]:
## 4 ##

# Build a single daily CSV joined with official Water Year (Oct→Sep)
# Inputs:
#   - outputs/dayflow_1929_2024.csv    (Date, NDOI)
#   - outputs/water_year_type.csv      (WY, Sac_Type, SJV_Type)
# Output: outputs/dayflow_wyt_daily.csv    (WY, NDOI, Sac_Type, SJV_Type, Date)



import os
import pandas as pd

DAYFLOW_CSV = os.path.join("outputs", "dayflow_1929_present.csv")
WYT_CSV     = os.path.join("outputs", "water_year_type.csv")
OUT_CSV     = os.path.join("outputs", "dayflow_wyt_daily.csv")

if not os.path.exists(DAYFLOW_CSV):
    raise SystemExit(f"Missing {DAYFLOW_CSV} — run chunk 2 first.")
if not os.path.exists(WYT_CSV):
    raise SystemExit(f"Missing {WYT_CSV} — run chunk 3 first.")

df_day = pd.read_csv(DAYFLOW_CSV, parse_dates=["Date"])
df_wyt = pd.read_csv(WYT_CSV)

df_day = df_day.dropna(subset=["Date"]).copy()
df_day["Date"] = pd.to_datetime(df_day["Date"]).dt.normalize()
df_day["NDOI"] = pd.to_numeric(df_day["NDOI"], errors="coerce")

# Official Water Year: add +3 months so Oct–Dec map to next year; then take calendar year
df_day["WY"] = (df_day["Date"] + pd.DateOffset(months=3)).dt.year.astype(int)

def normalize_wyt_col(s: pd.Series) -> pd.Series:
    m = {
        "WET": "W",
        "ABOVE NORMAL": "AN",
        "BELOW NORMAL": "BN",
        "DRY": "D",
        "CRITICAL": "C",
    }
    out = s.astype(str).str.strip()
    upper = out.str.upper()
    return upper.map(m).fillna(upper)

df_wyt["WY"] = pd.to_numeric(df_wyt["WY"], errors="coerce").astype("Int64")
df_wyt = df_wyt.dropna(subset=["WY"]).copy()
df_wyt["WY"] = df_wyt["WY"].astype(int)
df_wyt["Sac_Type"] = normalize_wyt_col(df_wyt["Sac_Type"])
df_wyt["SJV_Type"] = normalize_wyt_col(df_wyt["SJV_Type"])

joined = df_day.merge(df_wyt, on="WY", how="left")

joined = joined[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].copy()
joined["Date"] = joined["Date"].dt.strftime("%Y-%m-%d")

os.makedirs("outputs", exist_ok=True)
joined.to_csv(OUT_CSV, index=False)
print(f"✔ Wrote {OUT_CSV} — rows: {len(joined):,}")


✔ Wrote outputs/dayflow_wyt_daily.csv — rows: 34,699


In [54]:
## 5 ##

# Interactive NDOI + Water Year Type Query (from unified CSV created in 3.1)
# Uses the pre-joined file "dayflow_wyt_daily.csv" (created in chunk 3.1),
# which already contains daily Net Delta Outflow (NDOI), Sacramento and
# San Joaquin Water-Year-Type codes, and the official Water Year (WY).
# Prompts the user for NDOI range and optional WYT codes, shows all
# matching rows with the same columns and order as the source CSV
# (WY, NDOI, Sac_Type, SJV_Type, Date), saves all matches to
# outputs/match_results.csv, and lets the user pick a subset of dates
# to save for downstream Landsat steps at inputs/target_dates.csv.




import os, sys, textwrap
import pandas as pd

JOINED_CSV = os.path.join("outputs", "dayflow_wyt_daily.csv")

if not os.path.exists(JOINED_CSV):
    sys.exit("outputs/dayflow_wyt_daily.csv not found — run chunk 3.1 first!")

df = pd.read_csv(JOINED_CSV, parse_dates=["Date"])
df = df[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].copy()

min_wy, max_wy = int(df["WY"].min()), int(df["WY"].max())
min_yr = df["Date"].min().year
max_yr = df["Date"].max().year

EXPL = textwrap.dedent(f"""
    WATER-YEAR TYPE (WYT)
      Hydrologic class assigned by DWR for each water-year (Oct → Sep):
        W  = Wet
        AN = Above Normal
        BN = Below Normal
        D  = Dry
        C  = Critical
      Data available: from {min_wy} to {max_wy}

    NDOI (Net Delta Outflow Index)
      Daily net freshwater outflow from the legal Delta toward Suisun Bay.
      Units: cubic-feet-per-second (cfs).
      Daily data available: from October/01/{min_yr} to September/30/{max_yr}
""")
print(EXPL)

VALID_WYT = {"W", "AN", "BN", "D", "C"}

# Reference-only SQL (kept to preserve the original query structure, not executed)
sql_reference = """
    SELECT Date, NDOI,
           Sac_Type AS Sac_WYT,
           SJV_Type AS SJV_WYT
    FROM   v_dayflow_wyt
    WHERE  NDOI BETWEEN ? AND ?
      AND  (? IS NULL OR UPPER(TRIM(Sac_Type)) = UPPER(?))
      AND  (? IS NULL OR UPPER(TRIM(SJV_Type)) = UPPER(?))
    ORDER BY Date;
"""

while True:
    try:
        sac = input("Enter Sacramento WYT [W/AN/BN/D/C] (blank = any): ").strip().upper() or None
        sjv = input("Enter San Joaquin WYT [W/AN/BN/D/C] (blank = any): ").strip().upper() or None

        min_txt = input("Enter minimum NDOI (cfs) [blank = no limit]: ").strip()
        max_txt = input("Enter maximum NDOI (cfs) [blank = no limit]: ").strip()
        min_ndoi = float(min_txt) if min_txt else -1e12
        max_ndoi = float(max_txt) if max_txt else  1e12
        if min_ndoi > max_ndoi:
            min_ndoi, max_ndoi = max_ndoi, min_ndoi

        df_f = df[
            (df["NDOI"].between(min_ndoi, max_ndoi, inclusive="both")) &
            (True if sac is None else df["Sac_Type"].astype(str).str.strip().str.upper() == sac) &
            (True if sjv is None else df["SJV_Type"].astype(str).str.strip().str.upper() == sjv)
        ].copy()

        df_f = df_f.sort_values("Date")

        print(f"\nMatches: {len(df_f):,} days")

        if not df_f.empty:
            os.makedirs("outputs", exist_ok=True)
            df_f[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].to_csv("outputs/match_results.csv", index=False)
            print("✓ Saved ALL matches to outputs/match_results.csv")

            print("\nAll matching rows:\n")
            print(df_f[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].to_string(index=False))

            choice = input(
                "\nPick dates for satellite search "
                "(all / none / comma-separated list / start:end): "
            ).strip().lower()

            chosen = df_f.copy()
            if choice == "none":
                chosen = chosen.iloc[0:0]
            elif choice == "all" or choice == "":
                pass
            elif ":" in choice:
                try:
                    a_str, b_str = choice.split(":", 1)
                    a = pd.to_datetime(a_str).date()
                    b = pd.to_datetime(b_str).date()
                    if a > b: a, b = b, a
                    mask = (chosen["Date"].dt.date >= a) & (chosen["Date"].dt.date <= b)
                    chosen = chosen.loc[mask]
                except Exception as e:
                    print("⚠ Could not parse range; keeping all matches.", e)
            else:
                want, misses = [], []
                all_days = set(df_f["Date"].dt.date)
                for tok in choice.split(","):
                    tok = tok.strip()
                    if not tok:
                        continue
                    try:
                        d = pd.to_datetime(tok).date()
                        (want if d in all_days else misses).append(tok)
                    except Exception:
                        misses.append(tok)
                chosen = chosen[chosen["Date"].dt.date.isin(pd.to_datetime(want).date)] if want else chosen.iloc[0:0]
                if misses:
                    print("Note: ignored (not in matches):", ", ".join(misses))

            os.makedirs("inputs", exist_ok=True)
            chosen.sort_values("Date").to_csv("inputs/target_dates.csv", index=False)
            print(f"✓ Saved selected dates to inputs/target_dates.csv — {len(chosen)} rows")
            if len(chosen):
                print("\nSelected dates (first 20):")
                print(chosen.head(20)[["WY", "NDOI", "Sac_Type", "SJV_Type", "Date"]].to_string(index=False))
            else:
                print("\nNo dates selected.")
        else:
            print("No dates satisfy those conditions.")

    except Exception as e:
        print("⚠", e)

    again = input("\nRefine the search? (y/n): ").strip().lower()
    if again != "y":
        break



WATER-YEAR TYPE (WYT)
  Hydrologic class assigned by DWR for each water-year (Oct → Sep):
    W  = Wet
    AN = Above Normal
    BN = Below Normal
    D  = Dry
    C  = Critical
  Data available: from 1930 to 2024

NDOI (Net Delta Outflow Index)
  Daily net freshwater outflow from the legal Delta toward Suisun Bay.
  Units: cubic-feet-per-second (cfs).
  Daily data available: from October/01/1929 to September/30/2024



Enter Sacramento WYT [W/AN/BN/D/C] (blank = any):  w
Enter San Joaquin WYT [W/AN/BN/D/C] (blank = any):  w
Enter minimum NDOI (cfs) [blank = no limit]:  200000
Enter maximum NDOI (cfs) [blank = no limit]:  210000



Matches: 51 days
✓ Saved ALL matches to outputs/match_results.csv

All matching rows:

  WY     NDOI Sac_Type SJV_Type       Date
1938 203451.0        W        W 1938-03-22
1938 200964.0        W        W 1938-03-23
1941 203253.0        W        W 1941-01-28
1941 203494.0        W        W 1941-03-02
1941 200706.0        W        W 1941-04-08
1943 209210.0        W        W 1943-03-11
1943 202132.0        W        W 1943-03-12
1956 200840.0        W        W 1955-12-22
1956 202293.0        W        W 1956-01-21
1956 204641.0        W        W 1956-01-26
1956 204121.0        W        W 1956-01-29
1958 205760.0        W        W 1958-02-19
1958 201499.0        W        W 1958-04-02
1958 203091.0        W        W 1958-04-10
1965 203799.0        W        W 1964-12-30
1969 205885.0        W        W 1969-01-31
1969 207678.0        W        W 1969-02-17
1969 205489.0        W        W 1969-02-18
1974 202135.0        W        W 1974-01-18
1974 209389.0        W        W 1974-04-06
1982 2065


Pick dates for satellite search (all / none / comma-separated list / start:end):  all


✓ Saved selected dates to inputs/target_dates.csv — 51 rows

Selected dates (first 20):
  WY     NDOI Sac_Type SJV_Type       Date
1938 203451.0        W        W 1938-03-22
1938 200964.0        W        W 1938-03-23
1941 203253.0        W        W 1941-01-28
1941 203494.0        W        W 1941-03-02
1941 200706.0        W        W 1941-04-08
1943 209210.0        W        W 1943-03-11
1943 202132.0        W        W 1943-03-12
1956 200840.0        W        W 1955-12-22
1956 202293.0        W        W 1956-01-21
1956 204641.0        W        W 1956-01-26
1956 204121.0        W        W 1956-01-29
1958 205760.0        W        W 1958-02-19
1958 201499.0        W        W 1958-04-02
1958 203091.0        W        W 1958-04-10
1965 203799.0        W        W 1964-12-30
1969 205885.0        W        W 1969-01-31
1969 207678.0        W        W 1969-02-17
1969 205489.0        W        W 1969-02-18
1974 202135.0        W        W 1974-01-18
1974 209389.0        W        W 1974-04-06



Refine the search? (y/n):  n


In [55]:
## 6 ##
# Pair coverage check: requires BOTH tiles for each target date
# - Enforces WRS_PATH=44 and WRS_ROWS={33,34}
# - NO cloud cover filter (CLOUD_COVER_MAX=None)
# Writes:
#   outputs/satellite_coverage_report_pairs.csv
#   inputs/target_dates_effective_tiles.csv

import pandas as pd
import datacube

# Init
dc = datacube.Datacube()
bbox = get_bbox_wgs84()

# Detect products
st_products = list_landsat_st_products(dc)
if not st_products:
    raise RuntimeError("No Landsat ST products detected in this ODC index.")

print("Detected Landsat ST products:")
for p in st_products:
    print(" -", p)

# Load targets
sel = pd.read_csv(TARGETS_CSV, parse_dates=["Date"])
target_dates = sorted({pd.Timestamp(d).date() for d in sel["Date"].dropna()})
print(f"\nLoaded {len(target_dates)} target dates from {TARGETS_CSV}")
print(f"Searching ±{SEARCH_WINDOW_DAYS} days; requiring Path {WRS_PATH} and Rows {sorted(WRS_ROWS)}")
print(f"Cloud cover filter: {'OFF' if CLOUD_COVER_MAX is None else CLOUD_COVER_MAX}\n")

def find_candidates_for_date(t0, t1):
    """
    Return list of candidate dicts for all products in time window
    filtered to the needed WRS path/rows.
    """
    out = []
    for prod in st_products:
        dss = dc.find_datasets(product=prod, time=(t0, t1), **bbox)
        for ds in dss:
            dt = _extract_scene_datetime(ds)
            if dt is None:
                continue
            scene_day = _as_utc_naive(dt).normalize()

            path, row = _extract_wrs_path_row(ds)
            if path is None or row is None:
                continue
            if int(path) != int(WRS_PATH):
                continue
            if int(row) not in WRS_ROWS:
                continue

            cc = _extract_cloud_cover(ds)
            # Cloud filter intentionally OFF unless CLOUD_COVER_MAX is set
            if (CLOUD_COVER_MAX is not None) and (cc is not None) and (cc > float(CLOUD_COVER_MAX)):
                continue

            out.append({
                "Product": prod,
                "SceneDate": scene_day.date().isoformat(),
                "SceneDT": scene_day,
                "WRS_PATH": int(path),
                "WRS_ROW": int(row),
                "CloudCover": cc,
                "ODC_id": str(ds.id),
            })
    return out

def build_pairs(cands):
    """
    Group candidates by (Product, SceneDT) and keep only groups with BOTH required rows.
    Returns list of pairs with ODC ids for row33 and row34.
    """
    if not cands:
        return []

    df = pd.DataFrame(cands)
    if df.empty:
        return []

    pairs = []
    for (prod, scenedt), g in df.groupby(["Product", "SceneDT"]):
        rows_present = set(g["WRS_ROW"].tolist())
        if not WRS_ROWS.issubset(rows_present):
            continue

        # Grab exact ids for each row
        id_row = {}
        cc_row = {}
        for _, r in g.iterrows():
            rr = int(r["WRS_ROW"])
            # if duplicates exist, keep a stable choice:
            # choose the one with lowest cloud cover (if present), else first
            if rr not in id_row:
                id_row[rr] = str(r["ODC_id"])
                cc_row[rr] = r["CloudCover"]
            else:
                old_cc = cc_row[rr]
                new_cc = r["CloudCover"]
                if old_cc is None and new_cc is not None:
                    id_row[rr] = str(r["ODC_id"])
                    cc_row[rr] = new_cc
                elif (old_cc is not None) and (new_cc is not None) and (new_cc < old_cc):
                    id_row[rr] = str(r["ODC_id"])
                    cc_row[rr] = new_cc

        pairs.append({
            "Product": prod,
            "SceneDT": scenedt,
            "SceneDate": pd.Timestamp(scenedt).date().isoformat(),
            "ODC_id_row33": id_row[33],
            "ODC_id_row34": id_row[34],
            "CloudCover_row33": cc_row.get(33, None),
            "CloudCover_row34": cc_row.get(34, None),
        })

    return pairs

def choose_best_pair(target_date, pairs):
    """
    Choose best pair by:
      1) abs(offset days)
      2) product priority (only tiebreaker)
      3) prefer after (offset>=0) in ties
      4) stable tie-breakers
    """
    if not pairs:
        return None

    target = _as_utc_naive(pd.Timestamp(target_date).normalize())

    def score(p):
        off = int((p["SceneDT"] - target).days)
        return (
            abs(off),
            _product_priority(p["Product"]),
            0 if off >= 0 else 1,
            p["Product"],
            p["SceneDate"],
            p["ODC_id_row33"],
            p["ODC_id_row34"],
        )

    best = min(pairs, key=score)
    off = int((best["SceneDT"] - target).days)

    return {
        "TargetDate": pd.Timestamp(target_date).date().isoformat(),
        "Product": best["Product"],
        "SceneDate": best["SceneDate"],
        "OffsetDays": off,
        "WRS_PATH": WRS_PATH,
        "ODC_id_row33": best["ODC_id_row33"],
        "ODC_id_row34": best["ODC_id_row34"],
        "CloudCover_row33": best["CloudCover_row33"],
        "CloudCover_row34": best["CloudCover_row34"],
    }

rows_cov = []
rows_eff = []

for td in target_dates:
    t0, t1 = _date_range(td, SEARCH_WINDOW_DAYS)
    cands = find_candidates_for_date(t0, t1)
    pairs = build_pairs(cands)
    best = choose_best_pair(td, pairs)

    rows_cov.append({
        "target_date": td.isoformat(),
        "num_candidates_wrs_filtered": len(cands),
        "num_valid_pairs": len(pairs),
        "chosen_product": "" if best is None else best["Product"],
        "chosen_scene_date": "" if best is None else best["SceneDate"],
        "chosen_offset_days": "" if best is None else best["OffsetDays"],
        "chosen_odc_id_row33": "" if best is None else best["ODC_id_row33"],
        "chosen_odc_id_row34": "" if best is None else best["ODC_id_row34"],
    })

    if best is not None:
        rows_eff.append(best)

coverage = pd.DataFrame(rows_cov)
effective = pd.DataFrame(rows_eff)

OUTPUTS_DIR.mkdir(exist_ok=True)
coverage.to_csv(OUT_COVERAGE_PAIRS_CSV, index=False)
effective.to_csv(OUT_EFFECTIVE_TILES_CSV, index=False)

print(f"\n✓ Saved coverage report: {OUT_COVERAGE_PAIRS_CSV}")
print(f"✓ Saved effective tile pairs: {OUT_EFFECTIVE_TILES_CSV} ({len(effective)} rows)")
if effective.empty:
    print("⚠ No valid pairs found. This means ODC did not return BOTH rows 33 and 34 for your search window.")


AOI bbox (EPSG:4326): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
Detected Landsat ST products:
 - landsat9_c2l2_st
 - landsat8_c2l2_st
 - landsat7_c2l2_st
 - landsat5_c2l2_st

Loaded 51 target dates from inputs/target_dates.csv
Searching ±16 days; requiring Path 44 and Rows [33, 34]
Cloud cover filter: 18


✓ Saved coverage report: outputs/satellite_coverage_report_pairs.csv
✓ Saved effective tile pairs: inputs/target_dates_effective_tiles.csv (14 rows)


In [56]:
## 7 (PAIR) ##
# Scene metadata extraction for BOTH tiles (row33 + row34)
# Reads:  inputs/target_dates_effective_tiles.csv
# Writes: outputs/scene_metadata_odc_pairs.csv

import pandas as pd
import datacube

dc = datacube.Datacube()
bbox = get_bbox_wgs84()

EFFECTIVE_CSV = INPUTS_DIR / "target_dates_effective_tiles.csv"
OUT_META_CSV  = OUTPUTS_DIR / "scene_metadata_odc_pairs.csv"

eff = pd.read_csv(EFFECTIVE_CSV)
if eff.empty:
    raise ValueError(f"{EFFECTIVE_CSV} is empty (run chunk 6 first).")

required = {"TargetDate", "Product", "SceneDate", "OffsetDays", "ODC_id_row33", "ODC_id_row34"}
missing = required - set(eff.columns)
if missing:
    raise ValueError(f"Missing columns in {EFFECTIVE_CSV}: {missing}")

def _safe_props(ds):
    md = getattr(ds, "metadata_doc", None) or {}
    if isinstance(md, dict):
        return md.get("properties", {}) or {}
    return {}

rows = []

for _, r in eff.iterrows():
    product = str(r["Product"])
    scene_date = pd.Timestamp(r["SceneDate"]).normalize()
    t0 = scene_date
    t1 = t0 + pd.Timedelta(days=1)

    wanted33 = str(r["ODC_id_row33"])
    wanted34 = str(r["ODC_id_row34"])

    dss = dc.find_datasets(product=product, time=(t0, t1), **bbox)
    if not dss:
        continue

    ds33 = None
    ds34 = None
    for ds in dss:
        if str(ds.id) == wanted33:
            ds33 = ds
        elif str(ds.id) == wanted34:
            ds34 = ds

    # Fallback if not found (should be rare)
    if ds33 is None:
        ds33 = dss[0]
    if ds34 is None:
        ds34 = dss[-1] if len(dss) > 1 else dss[0]

    for label, ds in [("row33", ds33), ("row34", ds34)]:
        props = _safe_props(ds)
        path, row = _extract_wrs_path_row(ds)

        rows.append({
            "TargetDate": str(r["TargetDate"]),
            "Product": product,
            "SceneDate": scene_date.date().isoformat(),
            "OffsetDays": int(r["OffsetDays"]),
            "tile_label": label,
            "ODC_id": str(ds.id),
            "landsat:scene_id": props.get("landsat:scene_id", None),
            "datetime": props.get("datetime", None),
            "eo:cloud_cover": props.get("eo:cloud_cover", None),
            "landsat:wrs_path": path,
            "landsat:wrs_row": row,
        })

meta = pd.DataFrame(rows)
OUTPUTS_DIR.mkdir(exist_ok=True)
meta.to_csv(OUT_META_CSV, index=False)
print(f"✓ Saved scene metadata (pairs) to {OUT_META_CSV} ({len(meta)} rows)")


AOI bbox (EPSG:4326): {'x': (np.float64(-121.9404544080094), np.float64(-121.19670027286202)), 'y': (np.float64(37.62499087712795), np.float64(38.58916212265879))}
✓ Saved scene metadata (pairs) to outputs/scene_metadata_odc_pairs.csv (28 rows)


In [57]:
## 8A ##
from pathlib import Path
import os, zipfile, uuid
import numpy as np
import pandas as pd
import datacube
import xarray as xr
import rioxarray
from rioxarray.merge import merge_arrays

import rasterio
from rasterio.env import Env

EFFECTIVE_TILES_CSV = Path("inputs/target_dates_effective_tiles.csv")

OUT_DIR_TIFS  = Path("outputs/mosaicos_odc_lst_delta")
OUT_ZIP       = Path("outputs/mosaicos/mosaicos_celsius_odc_delta.zip")
OUT_SUMMARY   = Path("outputs/mosaicos_odc_lst_delta/run_summary_pairs.csv")

OUT_DIR_TIFS.mkdir(parents=True, exist_ok=True)
OUT_ZIP.parent.mkdir(parents=True, exist_ok=True)

NODATA_OUT = -9999.0

dc = datacube.Datacube()
aoi_out = get_aoi_in_crs(OUTPUT_CRS)

eff = pd.read_csv(EFFECTIVE_TILES_CSV)
if eff.empty:
    raise ValueError(f"{EFFECTIVE_TILES_CSV} is empty (run chunk 6 first).")

required_cols = {"TargetDate","Product","SceneDate","OffsetDays","ODC_id_row33","ODC_id_row34"}
missing = required_cols - set(eff.columns)
if missing:
    raise ValueError(f"Missing columns in {EFFECTIVE_TILES_CSV}: {missing}")

print(f"Loaded {len(eff)} effective TargetDate pairs from: {EFFECTIVE_TILES_CSV}")
print(f"Mask settings: WATER_ONLY={WATER_ONLY}  USE_CLEAR={USE_CLEAR}  USE_RADSAT_MASK={USE_RADSAT_MASK}")
print(f"Output folder: {OUT_DIR_TIFS}")


Loaded 14 effective TargetDate pairs from: inputs/target_dates_effective_tiles.csv
Mask settings: WATER_ONLY=True  USE_CLEAR=False  USE_RADSAT_MASK=False
Output folder: outputs/mosaicos_odc_lst_delta


In [58]:
## 8B ##
meas_cache = {}  # product -> (st_band, qa_pixel_band, qa_radsat_band)

def _get_ds_by_uuid(dc, id_str: str):
    try:
        ds_uuid = uuid.UUID(str(id_str))
    except Exception:
        raise ValueError(f"ODC_id is not a valid UUID: {id_str}")
    ds_obj = dc.index.datasets.get(ds_uuid)
    if ds_obj is None:
        raise RuntimeError(f"Could not fetch dataset by ODC_id from index: {id_str}")
    return ds_obj

def _finite_stats(da: xr.DataArray):
    v = da.values
    m = np.isfinite(v)
    n = int(m.sum())
    if n == 0:
        return 0, None, None, None
    vv = v[m]
    return n, float(vv.min()), float(vv.max()), float(vv.mean())

def _get_measurement_def(prod_def, band_name: str) -> dict:
    """
    ODC product definition measurements can be either:
      - dict: {"ST_B6": {...}, "QA_PIXEL": {...}}
      - list: [{"name": "ST_B6", ...}, {"name": "QA_PIXEL", ...}]
    This returns the dict for the measurement named `band_name` or {}.
    """
    meas = prod_def.get("measurements", None)
    if meas is None:
        return {}

    # Case 1: dict
    if isinstance(meas, dict):
        return meas.get(band_name, {}) or {}

    # Case 2: list
    if isinstance(meas, list):
        for item in meas:
            if isinstance(item, dict) and item.get("name") == band_name:
                return item
        return {}

    return {}

def _apply_scale_offset_to_kelvin(st_da: xr.DataArray, product: str, st_band: str) -> xr.DataArray:
    st_f = st_da.astype("float32")

    # 1) attrs
    scale = st_da.attrs.get("scale_factor", None)
    offset = st_da.attrs.get("add_offset", None)

    # 2) product definition fallback (handles dict OR list)
    prod = dc.index.products.get_by_name(product)
    mdef = {}
    if prod is not None:
        mdef = _get_measurement_def(prod.definition, st_band)

    if scale is None:
        scale = mdef.get("scale_factor", None)
    if offset is None:
        offset = mdef.get("add_offset", None)

    if (scale is not None) and (offset is not None):
        return st_f * float(scale) + float(offset)

    # Last resort: assume already Kelvin
    return st_f

def _st_to_celsius(st_da: xr.DataArray, product: str, st_band: str) -> xr.DataArray:
    kelvin = _apply_scale_offset_to_kelvin(st_da, product, st_band)
    return kelvin - 273.15

def _qa_bit_fractions(qa: xr.DataArray):
    frac_clear  = float(_bit_is_set(qa, BIT_CLEAR).mean().values)
    frac_water  = float(_bit_is_set(qa, BIT_WATER).mean().values)
    frac_cloud  = float(_bit_is_set(qa, BIT_CLOUD).mean().values)
    frac_shadow = float(_bit_is_set(qa, BIT_CLOUD_SHADOW).mean().values)
    frac_snow   = float(_bit_is_set(qa, BIT_SNOW).mean().values)
    return {
        "frac_clear": frac_clear,
        "frac_water": frac_water,
        "frac_cloud": frac_cloud,
        "frac_shadow": frac_shadow,
        "frac_snow": frac_snow,
    }

def _load_one_tile_with_diagnostics(ds_obj, product: str, label: str):
    if product not in meas_cache:
        meas_cache[product] = resolve_measurements_from_definition(dc, product)
    st_band, qa_pixel_band, qa_radsat_band = meas_cache[product]

    measurements = [st_band, qa_pixel_band]
    if (qa_radsat_band is not None) and USE_RADSAT_MASK:
        measurements.append(qa_radsat_band)

    with Env(AWS_REQUEST_PAYER="requester", GDAL_DISABLE_READDIR_ON_OPEN="YES"):
        ds = dc.load(
            datasets=[ds_obj],
            measurements=measurements,
            output_crs=OUTPUT_CRS,
            resolution=RESOLUTION,
            group_by="solar_day",
            skip_broken_datasets=True,
        )

    if ("time" not in ds.dims) or (ds.time.size == 0):
        diag = {
            "label": label,
            "status": "no_time",
            "st_band": st_band,
            "qa_pixel_band": qa_pixel_band,
            "qa_radsat_band": qa_radsat_band,
            "n_total": 0, "n_good": 0, "n_after_clip": 0
        }
        print(f"[{label}] WARNING: no time dimension after load")
        return None, diag

    st_raw = ds[st_band].isel(time=0)
    qa = ds[qa_pixel_band].isel(time=0)

    bits = _qa_bit_fractions(qa)
    print(
        f"[{label}] QA bit fractions: clear={bits['frac_clear']:.4f} water={bits['frac_water']:.4f} "
        f"cloud={bits['frac_cloud']:.4f} shadow={bits['frac_shadow']:.4f} snow={bits['frac_snow']:.4f}"
    )

    good = build_good_mask(qa)

    if USE_RADSAT_MASK and (qa_radsat_band is not None):
        sat = ds[qa_radsat_band].isel(time=0)
        good = good & (sat == 0)

    n_total, _, _, _ = _finite_stats(st_raw)
    st_masked = st_raw.where(good)
    n_good, _, _, _ = _finite_stats(st_masked)

    # Convert to Celsius using robust scale/offset fallback
    st_c = _st_to_celsius(st_masked, product, st_band)
    st_c = st_c.rio.write_crs(OUTPUT_CRS)

    st_clip = st_c.rio.clip(aoi_out.geometry, aoi_out.crs, drop=True)
    n_clip, tmin, tmax, tmean = _finite_stats(st_clip)

    # Scaling debug
    prod = dc.index.products.get_by_name(product)
    mdef = _get_measurement_def(prod.definition, st_band) if prod is not None else {}
    scale_attr = st_raw.attrs.get("scale_factor", None)
    off_attr   = st_raw.attrs.get("add_offset", None)
    scale_def  = mdef.get("scale_factor", None)
    off_def    = mdef.get("add_offset", None)

    diag = {
        "label": label,
        "status": "ok",
        "st_band": st_band,
        "qa_pixel_band": qa_pixel_band,
        "qa_radsat_band": qa_radsat_band,
        "n_total": n_total,
        "n_good": n_good,
        "n_after_clip": n_clip,
        "tmin_c": tmin,
        "tmax_c": tmax,
        "tmean_c": tmean,
        "scale_attr": scale_attr,
        "offset_attr": off_attr,
        "scale_def": scale_def,
        "offset_def": off_def,
        **bits,
    }

    print(
        f"[{label}] total={n_total:,}  good={n_good:,}  after_clip={n_clip:,}  "
        f"C(min/max/mean)={tmin}/{tmax}/{tmean}  "
        f"scale(attr/def)={scale_attr}/{scale_def}  offset(attr/def)={off_attr}/{off_def}"
    )

    if n_clip == 0:
        return None, diag

    return st_clip, diag


In [59]:
## 8C + 8D ##
run_rows = []

for _, r in eff.iterrows():
    target_date = str(r["TargetDate"])
    product     = str(r["Product"])
    scene_date  = str(r["SceneDate"])
    offset_days = int(r["OffsetDays"])
    id33        = str(r["ODC_id_row33"])
    id34        = str(r["ODC_id_row34"])

    print("\n" + "="*90)
    print(f"TargetDate={target_date}  SceneDate={scene_date}  Product={product}  OffsetDays={offset_days}")
    print(f"Row33={id33}")
    print(f"Row34={id34}")

    ds33 = _get_ds_by_uuid(dc, id33)
    ds34 = _get_ds_by_uuid(dc, id34)

    da33, d33 = _load_one_tile_with_diagnostics(ds33, product, "row33")
    da34, d34 = _load_one_tile_with_diagnostics(ds34, product, "row34")

    tiles = [t for t in (da33, da34) if t is not None]

    if len(tiles) == 0:
        print(f"⚠ SKIP {target_date}: 0 valid pixels after clip in BOTH tiles.")
        run_rows.append({
            "TargetDate": target_date,
            "SceneDate": scene_date,
            "Product": product,
            "OffsetDays": offset_days,
            "WRS_PATH": WRS_PATH,
            "ODC_id_row33": id33,
            "ODC_id_row34": id34,
            "OutTIF": "",
            "status": "skipped_empty_after_clip_both",
            "WATER_ONLY": WATER_ONLY,
            "USE_CLEAR": USE_CLEAR,
            "USE_RADSAT_MASK": USE_RADSAT_MASK,
            "OUTPUT_CRS": OUTPUT_CRS,
            "RESOLUTION": str(RESOLUTION),
            "row33_n_after_clip": d33.get("n_after_clip"),
            "row34_n_after_clip": d34.get("n_after_clip"),
        })
        continue

    # Prevent merge_arrays from introducing zeros
    tiles = [t.where(np.isfinite(t), np.nan) for t in tiles]

    mosaic = merge_arrays(tiles)
    mosaic = mosaic.rio.write_crs(OUTPUT_CRS)
    mosaic = mosaic.where(np.isfinite(mosaic), np.nan)

    n_m, tmin, tmax, tmean = _finite_stats(mosaic)
    print(f"[mosaic] valid={n_m:,}  C(min/max/mean)={tmin}/{tmax}/{tmean}")

    out_tif = OUT_DIR_TIFS / f"lst_delta_{target_date}_scene_{scene_date}_{product}_P{WRS_PATH}_R33R34.tif"

    # ---- WRITE (8D) ----
    mosaic_out = mosaic.astype("float32")
    mosaic_out = mosaic_out.where(np.isfinite(mosaic_out), NODATA_OUT)
    mosaic_out.rio.write_nodata(NODATA_OUT, inplace=True)

    mosaic_out.rio.to_raster(out_tif, nodata=NODATA_OUT)
    print(f"✓ Wrote: {out_tif}")

    run_rows.append({
        "TargetDate": target_date,
        "SceneDate": scene_date,
        "Product": product,
        "OffsetDays": offset_days,
        "WRS_PATH": WRS_PATH,
        "ODC_id_row33": id33,
        "ODC_id_row34": id34,
        "OutTIF": str(out_tif),
        "status": "written",
        "WATER_ONLY": WATER_ONLY,
        "USE_CLEAR": USE_CLEAR,
        "USE_RADSAT_MASK": USE_RADSAT_MASK,
        "OUTPUT_CRS": OUTPUT_CRS,
        "RESOLUTION": str(RESOLUTION),
        "mosaic_valid_px": n_m,
        "mosaic_tmin_c": tmin,
        "mosaic_tmax_c": tmax,
        "mosaic_tmean_c": tmean,
        "row33_n_after_clip": d33.get("n_after_clip"),
        "row34_n_after_clip": d34.get("n_after_clip"),
        "row33_frac_clear": d33.get("frac_clear"),
        "row33_frac_water": d33.get("frac_water"),
        "row34_frac_clear": d34.get("frac_clear"),
        "row34_frac_water": d34.get("frac_water"),
        "row33_scale_def": d33.get("scale_def"),
        "row33_offset_def": d33.get("offset_def"),
        "row34_scale_def": d34.get("scale_def"),
        "row34_offset_def": d34.get("offset_def"),
        "st_band": d33.get("st_band") or d34.get("st_band"),
        "qa_pixel_band": d33.get("qa_pixel_band") or d34.get("qa_pixel_band"),
        "qa_radsat_band": d33.get("qa_radsat_band") or d34.get("qa_radsat_band"),
    })



TargetDate=1986-03-09  SceneDate=1986-03-21  Product=landsat5_c2l2_st  OffsetDays=12
Row33=8bd39a83-0ca9-524e-b829-2c97e9775b05
Row34=33b32622-46fa-543e-8859-54c637b6dbc3
[row33] QA bit fractions: clear=0.6698 water=0.0286 cloud=0.0040 shadow=0.0061 snow=0.0162
[row33] total=51,288,458  good=1,463,044  after_clip=367,716  C(min/max/mean)=39113.8515625/43438.8515625/40187.2421875  scale(attr/def)=None/None  offset(attr/def)=None/None
[row34] QA bit fractions: clear=0.4145 water=0.2575 cloud=0.0409 shadow=0.0096 snow=0.0000
[row34] total=51,471,082  good=13,191,087  after_clip=271,365  C(min/max/mean)=38158.8515625/44432.8515625/39838.37890625  scale(attr/def)=None/None  offset(attr/def)=None/None
[mosaic] valid=586,315  C(min/max/mean)=0.0/43828.8515625/27829.537109375
✓ Wrote: outputs/mosaicos_odc_lst_delta/lst_delta_1986-03-09_scene_1986-03-21_landsat5_c2l2_st_P44_R33R34.tif

TargetDate=1986-03-18  SceneDate=1986-03-21  Product=landsat5_c2l2_st  OffsetDays=3
Row33=8bd39a83-0ca9-524e-

In [60]:
## 8E ##
summary = pd.DataFrame(run_rows)
summary.to_csv(OUT_SUMMARY, index=False)

written = summary.loc[summary["status"] == "written", "OutTIF"].tolist()

with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in written:
        if p and os.path.exists(p):
            z.write(p, arcname=os.path.basename(p))

print("\n" + "="*90)
print(f"✓ Saved run summary: {OUT_SUMMARY} (rows={len(summary)}, written={len(written)})")
print(f"✓ Saved zip: {OUT_ZIP}")
print(f"✓ Output folder: {OUT_DIR_TIFS}")



✓ Saved run summary: outputs/mosaicos_odc_lst_delta/run_summary_pairs.csv (rows=14, written=14)
✓ Saved zip: outputs/mosaicos/mosaicos_celsius_odc_delta.zip
✓ Output folder: outputs/mosaicos_odc_lst_delta
